In [468]:
import pandas as pd
import json
import os
import psycopg2

In [469]:
folder_path = "/Users/timurchiks/Downloads/settings"

rows = []
# "Сахар (общий)": {
#       "value": 0.494,
#       "unit": "g"
#     },
#     "Поваренная соль всего": {
#       "value": 0.353,
#       "unit": "g"
#     }
#     "Насыщенные жирные кислоты": {
#       "value": 0.878,
#       "unit": "g"
#     },

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        bls_code = filename.replace(".json", "")
        
        with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            data = data[0]
        
        ingredients = [ing["name"] for ing in data.get("ingredients", [])]
        techcard = data.get("technology", {})
        nutrition = data.get("nutrition_per_100g", {})
        protein      = nutrition.get("protein", {}).get("value", 0.0)
        fat          = nutrition.get("fat", {}).get("value", 0.0)
        carbohydrate = nutrition.get("carbohydrate", {}).get("value", 0.0)
        fiber        = nutrition.get("fiber", {}).get("value", 0.0)
        sugar        = nutrition.get("Сахар (общий)", {}).get("value", 0.0)
        salt_total   = nutrition.get("Поваренная соль всего", {}).get("value", 0.0)
        saturated_fat_mg= nutrition.get("Насыщенные жирные кислоты", {}).get("value", 0.0)
        kilocalories = data.get("calories_kcal_total_for_yield", 0.0)
        #sugar_mg, salt_total_mg, saturated_fat_mg
        rows.append({
            "bls_code":     bls_code,
            "ingredients":  ingredients,
            "techcard":     techcard,
            "kilocalories": kilocalories,
            "protein":      protein,
            "fat":          fat,
            "carbohydrate": carbohydrate,
            "fiber": fiber,
            "sugar_mg": sugar,
            "salt_total_mg": salt_total,
            "saturated_fat_mg": saturated_fat_mg,
        })

df_json = pd.DataFrame(rows)

In [470]:
with open("/Applications/python/nutristeppe/dish_category_from_4th_column.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df_json_codes = pd.DataFrame(data)

In [471]:
weights = pd.read_csv("/Applications/python/nutristeppe/weight.csv", sep=';')
weights

,code,Порция,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,GAR,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GAR01,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GAR0101,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GAR0102,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GAR0103,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
75,VEG02,250.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,BR,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,BR01,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,BR02,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [472]:
weights.rename(columns={"Порция": "serving_size_g"}, inplace=True)
weights.rename(columns={"code": "dish_category_code"}, inplace=True)
weights = weights[['dish_category_code', 'serving_size_g']]
weights

,dish_category_code,serving_size_g
0,GAR,150.0
1,GAR01,120.0
2,GAR0101,100.0
3,GAR0102,150.0
4,GAR0103,100.0
...,...,...
75,VEG02,250.0
76,BR,30.0
77,BR01,10.0
78,BR02,10.0


In [473]:
DB_CONFIG = {
    "host": "localhost",
    "database": "balaman",
    "user": "timurchiks",
    "password": "",
    "port": "5432"
}

conn = psycopg2.connect(**DB_CONFIG)
query = "select * from dishes;"
dishes = pd.read_sql(query, conn)
print(dishes)

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_3752/1801173792.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dishes = pd.read_sql(query, conn)


       dish_id bls_code                                               name  \
0        16564  F090400                           Смешанные сушеные фрукты   
1        16565  F090000                                   Смешанные фрукты   
2        16566  F090101                       Смешанные фрукты сырые целые   
3           19  X659012     "Schupfudel" (немецкая картофельная паста) (1)   
4           25  Y504322  "Wildkrustel" в панировке из дичи, обжаренная ...   
...        ...      ...                                                ...   
19320     9878  Y891312                                     Яблочные блины   
19321      405  Y891433                                Блинчики с творогом   
19322      899  Y892142                        Сладкие картофельные клецки   
19323     3078  Y912130   Мясная котлета с картофельным салатом и горчицей   
19324     1427  Y921062                         Пита с говядиной и салатом   

      description recipe_description  dish_category_id dish_cat

In [474]:
dishes = dishes[((dishes["type_5"] == 1) | (dishes["type_5"] == 2)) & ((dishes['type'] == 2) | (dishes['type'] == 3))]

In [475]:
dishes.shape

(6091, 27)

In [476]:
dishes.columns

Index(['dish_id', 'bls_code', 'name', 'description', 'recipe_description',
       'dish_category_id', 'dish_category_code', 'price', 'weight',
       'kilocalories', 'image_url', 'has_relation_with_products',
       'health_factor', 'created_at', 'updated_at', 'protein', 'fat',
       'carbohydrate', 'cfv', 'fvnl', 'index_health', 'type', 'type_5',
       'new_name', 'cooking_technology', 'serving_method',
       'quality_requirements'],
      dtype='object')

In [477]:
dishes['ingredients'] = None
dishes['techcard'] = None
dishes['fiber'] = None
dishes['sugar_mg'] = None
dishes['salt_total_mg'] = None
dishes['saturated_fat_mg'] = None
#sugar_mg, salt_total_mg, saturated_fat_mg

In [478]:
dishes.columns

Index(['dish_id', 'bls_code', 'name', 'description', 'recipe_description',
       'dish_category_id', 'dish_category_code', 'price', 'weight',
       'kilocalories', 'image_url', 'has_relation_with_products',
       'health_factor', 'created_at', 'updated_at', 'protein', 'fat',
       'carbohydrate', 'cfv', 'fvnl', 'index_health', 'type', 'type_5',
       'new_name', 'cooking_technology', 'serving_method',
       'quality_requirements', 'ingredients', 'techcard', 'fiber', 'sugar_mg',
       'salt_total_mg', 'saturated_fat_mg'],
      dtype='object')

In [479]:
dishes[dishes["bls_code"] == "Y995140"][['bls_code', 'ingredients', 'techcard', 'kilocalories', 'protein', 'fat', 'carbohydrate', "fiber"]]

,bls_code,ingredients,techcard,kilocalories,protein,fat,carbohydrate,fiber
10277,Y995140,None,None,83.86,0.41,0.23,20.04,None


In [480]:
dishes['pfs_unit'] = 'mg'

In [481]:
df_json['pfs_unit'] = 'g'

In [482]:
dishes = dishes.set_index("bls_code")
df_json = df_json.set_index("bls_code")
dishes.update(df_json)
df_last = dishes.reset_index()
df_last

,bls_code,dish_id,name,description,recipe_description,dish_category_id,dish_category_code,price,weight,kilocalories,...,cooking_technology,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,pfs_unit
0,X659012,19,"""Schupfudel"" (немецкая картофельная паста) (1)",None,None,5.0,GAR0201,0.0,200.0,100.0,...,None,None,None,None,None,None,None,None,None,mg
1,Y504322,25,"""Wildkrustel"" в панировке из дичи, обжаренная ...",None,None,7.0,MT01,0.0,300.0,141.0,...,None,None,None,None,None,None,None,None,None,mg
2,X880522,38,«Риси Писи» (горох в рисе) (2),None,None,6.0,GAR0202,0.0,250.0,115.0,...,None,None,None,None,None,None,None,None,None,mg
3,X693213,41,«Хоппель-Поппель» (омлет с жареным картофелем ...,None,None,41.0,EGG,0.0,350.0,131.0,...,None,None,None,None,None,None,None,None,None,mg
4,X468713,42,«Gaisburger Marsch» (картофель с говядиной) (1),None,None,29.0,CMT02,0.0,450.0,96.0,...,None,None,None,None,None,None,None,None,None,mg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6086,Y891312,9878,Яблочные блины,None,None,16.0,DGH0201,0.0,100.0,194.9,...,"Муку смешивают с сахаром и солью, отдельно взб...","Подают горячими, можно посыпать сахаром или по...","Блины должны быть золотистого цвета, мягкие и ...","[мука, яйцо, молоко, яблоко, сахар, соль, масл...","Муку смешивают с сахаром и солью, отдельно взб...",0.845,8.563,0.394,1.771,g
6087,Y891433,405,Блинчики с творогом,None,None,38.0,CC,0.0,100.0,188.6,...,"В глубокой миске смешать муку, яйца, молоко и ...","Подают горячими, можно добавить свежие фрукты.","Блинчики должны быть золотистого цвета, мягкие...","[мука, яйцо, молоко, творог, соль, масло расти...","В глубокой миске смешать муку, яйца, молоко и ...",0.604,7.466,0.274,2.559,g
6088,Y892142,899,Сладкие картофельные клецки,None,None,3.0,GAR0103,0.0,100.0,178.5,...,"Картофель отварить до полной готовности, воду ...","Подают горячими, можно посыпать сахаром или по...","Внешний вид: клецки одинаковой формы, без трещ...","[картофель, мука, сахар, соль, яйцо]","Картофель отварить до полной готовности, воду ...",1.66,3.734,0.32,0.215,g
6089,Y912130,3078,Мясная котлета с картофельным салатом и горчицей,None,None,10.0,MT04,0.0,100.0,123.9,...,Для приготовления котлеты говядину пропускаем ...,"Блюдо подают горячим, температура подачи 65-70°C.","Котлета должна быть золотистой, с хрустящей ко...","[говядина, картофель, лук репчатый, горчица, м...",Для приготовления котлеты говядину пропускаем ...,0.641,0.596,0.341,0.684,g


In [483]:
df_merged = df_last.merge(weights, on="dish_category_code", how="left")

In [484]:
df_merged.columns

Index(['bls_code', 'dish_id', 'name', 'description', 'recipe_description',
       'dish_category_id', 'dish_category_code', 'price', 'weight',
       'kilocalories', 'image_url', 'has_relation_with_products',
       'health_factor', 'created_at', 'updated_at', 'protein', 'fat',
       'carbohydrate', 'cfv', 'fvnl', 'index_health', 'type', 'type_5',
       'new_name', 'cooking_technology', 'serving_method',
       'quality_requirements', 'ingredients', 'techcard', 'fiber', 'sugar_mg',
       'salt_total_mg', 'saturated_fat_mg', 'pfs_unit', 'serving_size_g'],
      dtype='object')

In [485]:
df_merged=df_merged.drop(columns=['weight','image_url','name','dish_id','description','recipe_description','price','has_relation_with_products','health_factor','created_at','updated_at'])

In [486]:
df_merged.rename(columns={"new_name": "name", "type_5": "availability_type", "cooking_technology": "steps"}, inplace=True)

In [487]:
df_merged.to_csv('19March.csv')

In [488]:
df_last[df_last["bls_code"] == "Y995140"][['bls_code', 'ingredients', 'techcard', 'weight', 'kilocalories', 'protein', 'fat', 'carbohydrate', 'fiber']]

,bls_code,ingredients,techcard,weight,kilocalories,protein,fat,carbohydrate,fiber
2833,Y995140,"[груша, Белый сахар, Свежий лимонный сок]",Помыть и очистить груши от кожуры. Разрезать п...,100.0,83.86,0.406,0.232,20.04,2.24


In [489]:
df_last.shape

(6091, 34)

In [490]:
df_last.columns

Index(['bls_code', 'dish_id', 'name', 'description', 'recipe_description',
       'dish_category_id', 'dish_category_code', 'price', 'weight',
       'kilocalories', 'image_url', 'has_relation_with_products',
       'health_factor', 'created_at', 'updated_at', 'protein', 'fat',
       'carbohydrate', 'cfv', 'fvnl', 'index_health', 'type', 'type_5',
       'new_name', 'cooking_technology', 'serving_method',
       'quality_requirements', 'ingredients', 'techcard', 'fiber', 'sugar_mg',
       'salt_total_mg', 'saturated_fat_mg', 'pfs_unit'],
      dtype='object')

In [491]:
nutri_last = pd.read_csv('/Users/timurchiks/Downloads/newdb_13m_ready.csv',sep=';')
nutri_last

,bls_code,dish_name,category,energy_kcal,serving_size_g,water_mg,protein_mg,fat_mg,carbohydrates_mg,sugar_mg,...,докозагексаеновая кислота_100g,полиненасыщенные жирные кислоты_100g,омега-3 жирные кислоты_100g,омега-6 жирные кислоты_100g,холестерин_100g,хлебные единицы_100g,поваренная соль всего_100g,health_index,index_health,type_availability
0,Y363323,Свиная рулька с овощами и соусом,CMT02,140.20,100.0,0.745,10.620,9.562,2.909,2.305,...,0.000,1.489,0.408,1.081,0.037,0.239,0.386,3.3860,NaN,1.0
1,U501262,Запечённая нежирная свинина,MT02,136.60,100.0,0.711,20.600,5.444,1.287,0.668,...,0.000,0.605,0.064,0.541,0.067,0.107,0.408,4.3405,NaN,1.0
2,Y221232,"Телячья отбивная без панировки, жареная",MT02,113.80,100.0,0.727,18.030,4.499,0.287,0.087,...,0.000,0.941,0.244,0.696,0.068,0.024,0.412,4.1133,NaN,1.0
3,U511262,Филе свинины нежирное запеченное,MT02,121.60,100.0,0.731,21.490,3.952,0.002,0.002,...,0.000,0.737,0.216,0.521,0.054,0.000,0.408,4.4541,NaN,1.0
4,16-041,Филе камбалы на пару,FSH,63.88,100.0,0.830,11.710,1.840,0.125,0.122,...,0.101,0.366,0.003,0.007,0.042,0.009,0.296,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5328,M780400,Кулинарный сыр (немецкий) не менее 30% жирности,CH01,166.00,30.0,70.130,13.011,11.000,3.488,3.488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,2.0
5329,M780500,Немецкий варёный сыр (жирность от 40%),CH01,187.00,30.0,68.355,12.000,13.900,3.400,3.400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,2.0
5330,M780600,"Немецкий варёный сыр, 45% жирности",CH01,219.00,30.0,64.656,12.400,17.399,3.200,3.200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,2.0
5331,M780700,Немецкий вареный сыр 50% жирности,CH01,236.00,30.0,61.520,11.000,19.000,5.400,5.400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0


In [492]:
l=nutri_last.columns.to_list()

In [493]:
print(l)

['bls_code', 'dish_name', 'category', 'energy_kcal', 'serving_size_g', 'water_mg', 'protein_mg', 'fat_mg', 'carbohydrates_mg', 'sugar_mg', 'saturated_fat_mg', 'salt_total_mg', 'leucine_mg', 'lysine_mg', 'methionine_mg', 'cysteine_mg', 'phenylalanine_mg', 'tyrosine_mg', 'threonine_mg', 'tryptophan_mg', 'valine_mg', 'essential_amino_acids_mg', 'dha_docosahexaenoic_acid_mg', 'pufa_mg', 'omega3_mg', 'omega6_mg', 'cholesterol_mg', 'bread_units', 'allergens', 'total_net_weight_g', 'steps', 'ingredients', 'vitamina', 'vitamind', 'vitamine', 'vitamink', 'vitaminb1', 'vitaminb2', 'vitaminb3', 'vitaminb5', 'vitaminb6', 'vitaminb7', 'vitaminb9', 'vitaminb12', 'vitaminc', 'fiber', 'potassium', 'calcium', 'magnesium', 'phosphorus', 'iron', 'zinc', 'iodine', 'sodium', 'сахар (общий)_100g', 'насыщенные жирные кислоты_100g', 'water_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'vitamina_100g', 'vitamind_100g', 'vitamine_100g', 'vitamink_100g', 'vitaminb1_100g', 'vitaminb2_100g', 'vitaminb3_1